# 06 - SQL and Trend Questions

This notebook introduces V3 of the Notely AI PM Analytics Agent: standard metric trend analysis.

The immediate failure case was a stakeholder-style question: `help me check each week's activation rate from April to June and analyze if there is any increasing trend`. The early agent treated week as a `group_by` idea, but week is a time grain, not a segment dimension.

## What V3 Solves

V3 gives the agent a safe way to answer questions like:

- weekly activation rate from April to June
- monthly paid conversion trend
- daily upload completion during an incident
- weekly summaries per active user by platform

These are standard metrics over time. They should use trusted metric definitions, not free-form SQL invented by the model.

## V3 vs V4

V3 is standard metric time-series analysis. It uses tools such as `get_metric_timeseries` and `analyze_metric_trend`.

V4 will be a safer ad-hoc SQL analyst. That future layer will let the agent inspect schemas, write SELECT-only SQL, execute it, and explain the results.

The reason to build V3 first is stability: many PM questions are about standard metrics over time, and those can be answered with trusted metric logic.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

## Tool 1: get_metric_timeseries

This tool returns the raw trusted metric values by day, week, or month. It is useful when the user asks to see a table of metric values over time.

In [ ]:
from agent.tools import get_metric_timeseries, analyze_metric_trend

weekly_activation = get_metric_timeseries(
    metric_name='activation_rate',
    start_date='2026-04-01',
    end_date='2026-06-30',
    grain='week',
)
weekly_activation['rows'][:5]

## Tool 2: analyze_metric_trend

This tool wraps the time series and adds a small trend summary: first value, last value, relative change, highest period, lowest period, and direction.

In [ ]:
trend = analyze_metric_trend(
    metric_name='activation_rate',
    start_date='2026-04-01',
    end_date='2026-06-30',
    grain='week',
)
trend['trend_summary']

In [ ]:
trend['timeseries']['rows']

## Segment Example

The same trend tool can filter to a segment. For example, weekly iOS activation rate uses `segment='iOS'`. The tool infers `group_by='platform'` for common segment names.

In [ ]:
ios_trend = analyze_metric_trend(
    metric_name='activation_rate',
    start_date='2026-04-01',
    end_date='2026-06-30',
    grain='week',
    segment='iOS',
)
ios_trend['trend_summary']

## Rerun The Agent

From Terminal, test the original failure case:

```bash
python3 llm_agent_demo.py --show-trace "help me check each week's activation rate from April to June and help me analyze if there is any increasing trend"
```

A better trace should call `analyze_metric_trend`, not `get_metric` with `group_by='week'`.

## What To Look For In The Output

Checklist:

- Did it call `analyze_metric_trend`?
- Did it use `grain='week'`?
- Did it avoid saying `group_by` is unsupported?
- Did it provide weekly values or a summary based on the time series?
- Did it describe the trend direction with caveats if the pattern is mixed?